# HW1. Оценка сохраненных моделей на новых данных

Ноутбук делает то же, что `solution.ipynb` из примера преподавателя: загружает **уже обученные модели** из `artifacts/`,
предсказывает на диалогах из `hidden_test.jsonl`, сравнивает ответы с `hidden_labels.jsonl` и считает метрики обеих задач.
Ответы всех моделей сохраняются в csv (`dialog_id`, `task`, `model`, `prediction`).

**Что нужно рядом с ноутбуком**

| Файл | Что это |
|:--|:--|
| `solution_utils.py` | ваш файл с функциями (из него берутся `read_dialogs`, `build_frames` и `prepare_topic_texts`) |
| `artifacts/*.joblib` | модели задачи 1 (реакция) |
| `artifacts/topics/*.joblib` | модели задачи 2 (темы) |
| `hidden_test.jsonl`, `hidden_labels.jsonl` | новые данные и ответы к ним |

Пути можно поменять переменными окружения `HW1_DATA`, `HW1_LABELS`, `HW1_PREDICTIONS`, `HW1_ARTIFACTS`, как в примере.
Модели не переобучаются: ничего из ноутбука не пишет в `artifacts/`.

In [ ]:
import json
import os
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, balanced_accuracy_score, confusion_matrix, f1_score,
    homogeneity_completeness_v_measure, normalized_mutual_info_score, precision_recall_fscore_support,
)
from sklearn.metrics.cluster import contingency_matrix

import solution_utils as su

In [ ]:
DATA_PATH = os.environ.get("HW1_DATA", "hidden_test.jsonl")
LABELS_PATH = os.environ.get("HW1_LABELS", "hidden_labels.jsonl")
PREDICTIONS_PATH = os.environ.get("HW1_PREDICTIONS", "predictions.csv")
ARTIFACTS_DIR = Path(os.environ.get("HW1_ARTIFACTS", "artifacts"))

assert os.path.exists(DATA_PATH), f"нет файла с диалогами: {DATA_PATH}"
assert not LABELS_PATH or os.path.exists(LABELS_PATH), f"нет файла с ответами: {LABELS_PATH}"

dialogs = su.read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))

## Функции для инференса и метрик

Эти функции нужны, чтобы старые модели (sklearn-пайплайны на тексте) и новые (классы из `solution_utils.py`) запускались одинаково.
Для итогового решения их надо перенести в `solution_utils.py`: ноутбук в примере только импортирует функции оттуда.

* `smart_tokenizer` — **копия** функции из ноутбука обучения. Модель `kirp_model` сохранена как «ссылка на функцию `smart_tokenizer` в ноутбуке»,
  и без этой функции в текущем ядре `joblib.load` не сможет ее загрузить. Код функции должен совпадать с тем, что был при обучении.
* `read_labels` понимает два формата файла с ответами: `{"dialog_id": ..., "labels": {"reaction": ..., "topic": ...}}`
  и плоский `{"dialog_id": ..., "reaction": ..., "topic": ...}`.

In [ ]:
CLASSES = ["negative", "neutral", "positive"]

# --- smart_tokenizer: копия из ноутбука обучения (нужна только для распаковки kirp_model) -------------
try:
    from nltk.corpus import stopwords
    from nltk.stem import SnowballStemmer

    _stemmer_en, _stemmer_ru = SnowballStemmer("english"), SnowballStemmer("russian")
    _all_stop_words = set(stopwords.words("english")) | set(stopwords.words("russian"))
    _WORD_REG = re.compile(r"[^\W\d_]+")

    def smart_tokenizer(text: str) -> list[str]:
        result = []
        for word in _WORD_REG.findall(text.lower()):
            if (len(word) < 2 or word in _all_stop_words) and (word != "no" or "ok" or "yes" or "not"):
                continue
            if "а" <= word[0] <= "я" or word[0] == "ё":
                result.append(_stemmer_ru.stem(word))
            else:
                result.append(_stemmer_en.stem(word))
        return result
except Exception as error:  # без nltk модели с токенизатором не загрузятся, остальные работают
    print("nltk недоступен, smart_tokenizer не определен:", error)


# --- чтение ответов -----------------------------------------------------------------------------
def read_labels(path: str, task: str) -> dict:
    # dialog_id -> метка для задачи task ("reaction" или "topic"); диалоги без метки пропускаются
    labels = {}
    with open(path, encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            record = json.loads(line)
            inner = record.get("labels")
            label = inner.get(task) if isinstance(inner, dict) else None
            if label is None:
                label = record.get(task)
            if label is not None:
                labels[record["dialog_id"]] = label
    return labels


# --- загрузка моделей ---------------------------------------------------------------------------
def load_models(directory: Path):
    # Все *.joblib из папки: ({имя файла без расширения: модель}, {имя: почему не загрузилась})
    models, failed = {}, {}
    for path in sorted(Path(directory).glob("*.joblib")):
        try:
            models[path.stem] = joblib.load(path)
        except Exception as error:
            failed[path.stem] = f"{type(error).__name__}: {str(error)[:150]}"
    return models, failed


# --- задача 1: инференс --------------------------------------------------------------------------
def extract_text(turns: list) -> str:
    # Как при обучении sklearn-пайплайнов: все реплики диалога одной строкой
    return " ".join((turn.get("text") or "") for turn in turns if isinstance(turn, dict))


def predict_reaction(model, dialogs: list) -> list:
    # Обычные sklearn-модели учились на тексте диалога, классы из solution_utils.py принимают сами диалоги
    data = [extract_text(d["turns"]) for d in dialogs] if isinstance(model, BaseEstimator) else dialogs
    return [str(answer) for answer in model.predict(data)]


# --- задача 2: тексты и инференс ----------------------------------------------------------------
def topic_texts(dialogs: list) -> pd.Series:
    # Тексты для моделей тем: та же подготовка, что при обучении (su.prepare_topic_texts), в порядке диалогов
    _, messages = su.build_frames(dialogs)
    prepared = su.prepare_topic_texts(messages)
    return prepared.reindex([d["dialog_id"] for d in dialogs]).fillna("")


def predict_topic(model, texts: pd.Series) -> list:
    return [str(answer) for answer in model.predict(texts)]


# --- метрики -------------------------------------------------------------------------------------
def labeled_pairs(dialogs: list, predictions: list, labels: dict):
    # Ответы и предсказания только для диалогов, у которых есть метка
    index = [i for i, d in enumerate(dialogs) if d["dialog_id"] in labels]
    return [labels[dialogs[i]["dialog_id"]] for i in index], [predictions[i] for i in index]


def reaction_metrics(y_true: list, y_pred: list) -> dict:
    precision, recall, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=CLASSES, zero_division=0)
    metrics = {
        "macro_f1": f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }
    for cls, p, r in zip(CLASSES, precision, recall):
        metrics[f"precision_{cls}"], metrics[f"recall_{cls}"] = p, r
    return metrics


def topic_metrics(y_true: list, y_pred: list) -> dict:
    # Названия тем у нас и у проверяющих не совпадают, поэтому сравниваются разбиения (как в README)
    homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y_true, y_pred)
    table = contingency_matrix(y_true, y_pred)  # строки — эталон, столбцы — наши темы
    return {
        "ARI": adjusted_rand_score(y_true, y_pred),
        "NMI": normalized_mutual_info_score(y_true, y_pred),
        "homogeneity": homogeneity,
        "completeness": completeness,
        "V": v_measure,
        "purity": table.max(axis=0).sum() / table.sum(),
        "тем у нас": len(set(y_pred)),
        "тем в эталоне": len(set(y_true)),
    }


# --- сохранение ответов --------------------------------------------------------------------------
def save_predictions(path: str, dialogs: list, predictions: dict) -> None:
    # predictions: {"reaction": {имя модели: ответы}, "topic": {...}} -> csv dialog_id, task, model, prediction
    rows = []
    for task, by_model in predictions.items():
        for name, answers in by_model.items():
            rows.extend({"dialog_id": d["dialog_id"], "task": task, "model": name, "prediction": a}
                        for d, a in zip(dialogs, answers))
    pd.DataFrame(rows, columns=["dialog_id", "task", "model", "prediction"]).to_csv(path, index=False)

## Ответы

Читаем `hidden_labels.jsonl` и проверяем, что метки нашлись: если ниже нули, значит файл с ответами в другом формате
(поправьте `read_labels`, она описана выше).

In [ ]:
reaction_labels = read_labels(LABELS_PATH, "reaction") if LABELS_PATH else {}
topic_labels = read_labels(LABELS_PATH, "topic") if LABELS_PATH else {}

ids = {d["dialog_id"] for d in dialogs}
print(f"меток reaction: {len(reaction_labels)}, меток topic: {len(topic_labels)}, диалогов в hidden_test: {len(dialogs)}")
print("метки есть у диалогов, которых нет в hidden_test:", len(set(reaction_labels) - ids), "(reaction),", len(set(topic_labels) - ids), "(topic)")
if reaction_labels:
    print("\nдоли классов реакции в новых данных:")
    print(pd.Series(reaction_labels).value_counts(normalize=True).round(3).to_string())

## Задача 1. Реакция пользователя

Загружаются все `*.joblib` из `artifacts/`. Если какая-то модель не загрузилась, причина будет в таблице ниже, а остальные посчитаются.

In [ ]:
reaction_models, reaction_failed = load_models(ARTIFACTS_DIR)

reaction_predictions, reaction_table = {}, {}
for name, model in reaction_models.items():
    try:
        reaction_predictions[name] = predict_reaction(model, dialogs)
        if reaction_labels:
            y_true, y_pred = labeled_pairs(dialogs, reaction_predictions[name], reaction_labels)
            reaction_table[name] = reaction_metrics(y_true, y_pred)
    except Exception as error:
        reaction_failed[name] = f"{type(error).__name__}: {str(error)[:150]}"
        reaction_predictions.pop(name, None)

print("загружено и отработало моделей реакции:", len(reaction_predictions))
if reaction_failed:
    print("\nне отработали:")
    display(pd.Series(reaction_failed, name="причина").to_frame())
(pd.DataFrame(reaction_table).T.round(3) if reaction_table
 else "метки reaction не найдены в файле ответов, метрики не считаются" if reaction_predictions
 else "ни одна модель реакции не отработала: причины выше (или в artifacts/ нет файлов *.joblib)")

### Матрица ошибок лучшей модели (по macro-F1)

In [ ]:
if reaction_table:
    best_reaction = pd.DataFrame(reaction_table).T["macro_f1"].idxmax()
    y_true, y_pred = labeled_pairs(dialogs, reaction_predictions[best_reaction], reaction_labels)
    print("лучшая модель:", best_reaction)
    display(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=CLASSES),
                         index=[f"факт {c}" for c in CLASSES], columns=[f"предсказано {c}" for c in CLASSES]))

## Задача 2. Темы

Модели из `artifacts/topics/`. Названия тем в `hidden_labels.jsonl` не совпадают с нашими, поэтому accuracy не считается: сравниваются разбиения
(ARI, NMI, однородность, полнота, V-мера, чистота).

In [ ]:
topic_models, topic_failed = load_models(ARTIFACTS_DIR / "topics")

topic_predictions, topic_table = {}, {}
if topic_models:
    texts = topic_texts(dialogs)
    for name, model in topic_models.items():
        try:
            topic_predictions[name] = predict_topic(model, texts)
            if topic_labels:
                y_true, y_pred = labeled_pairs(dialogs, topic_predictions[name], topic_labels)
                topic_table[name] = topic_metrics(y_true, y_pred)
        except Exception as error:
            topic_failed[name] = f"{type(error).__name__}: {str(error)[:150]}"
            topic_predictions.pop(name, None)

print("загружено и отработало моделей тем:", len(topic_predictions))
if topic_failed:
    print("\nне отработали:")
    display(pd.Series(topic_failed, name="причина").to_frame())
(pd.DataFrame(topic_table).T.round(3) if topic_table
 else "метки topic не найдены в файле ответов, метрики не считаются" if topic_predictions
 else "моделей тем нет или они не отработали (причины выше; модели лежат в artifacts/topics/*.joblib)")

### Как наши темы легли на темы проверяющих (лучшая модель по ARI)

Строки — эталонные темы, столбцы — наши. Видно, какие наши темы «склеили» или «раздробили» чужие.

In [ ]:
if topic_table:
    best_topic = pd.DataFrame(topic_table).T["ARI"].idxmax()
    y_true, y_pred = labeled_pairs(dialogs, topic_predictions[best_topic], topic_labels)
    print("лучшая модель по ARI:", best_topic)
    display(pd.crosstab(pd.Series(y_true, name="эталон"), pd.Series(y_pred, name="наша тема")))

## Ответы всех моделей

Последняя ячейка сохраняет ответы всех моделей обеих задач (`dialog_id`, `task`, `model`, `prediction`).

In [ ]:
save_predictions(PREDICTIONS_PATH, dialogs, {"reaction": reaction_predictions, "topic": topic_predictions})
n_models = len(reaction_predictions) + len(topic_predictions)
print("строк в файле ответов:", len(dialogs) * n_models, "->", PREDICTIONS_PATH)